In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1Srii8TYHgsmhniB8IbmQJ_kjXt_hVr__", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))


In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first!

import torch
import torch.nn as nn
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU available: {gpu_name}")
    print(f"   Memory: {gpu_mem_gb:.1f} GB")
else:
    device = torch.device('cpu')
    gpu_mem_gb = 0
    print("No GPU detected. Calculator still works with manual GPU spec.")

print(f"\nPython {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

np.random.seed(42)
%matplotlib inline

In [ ]:
#@title 🎧 Listen: Motivation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_motivation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# Will It Fit? A GPU Memory Calculator -- Vizuara

## 1. Why Does This Matter?

Before you spend hours downloading a model, setting up your training script, and launching a job -- only to hit OOM 30 seconds in -- you should be able to answer one question:

**"Will this model fit on my GPU for training?"**

In this notebook, we build a complete memory calculator that answers this question for **any** model configuration. It combines everything from Notebooks 1 and 2:
- Weights (fp16 or fp32)
- Gradients (same size as weights)
- Optimizer states (Adam: master + m + v)
- Activations (batch size, sequence length, layers)

**What you will build:** An interactive "Will It Fit?" tool that predicts memory usage and recommends the maximum batch size for your GPU.

In [ ]:
#@title 🎧 Listen: Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Building Intuition

Think of GPU memory like packing a suitcase for a trip:

- **Weights** = your clothes (fixed size, must bring them)
- **Gradients** = a duplicate set of clothes (same size as weights)
- **Optimizer states** = your toiletries, shoes, books (surprisingly large -- often bigger than the clothes!)
- **Activations** = souvenirs you pick up along the way (grows with trip length and how much you buy)

A good packer (engineer) checks if everything fits BEFORE trying to close the suitcase.

In [ ]:
#@title 🎧 Listen: Math
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_math.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. The Mathematics

The complete formula for training memory:

$$M_{\text{total}} = \underbrace{2\Psi}_{\text{weights}} + \underbrace{2\Psi}_{\text{gradients}} + \underbrace{12\Psi}_{\text{optimizer}} + \underbrace{L \cdot B \cdot S \cdot H \cdot 34}_{\text{activations}}$$

where $\Psi$ = number of parameters (mixed precision with Adam).

The first three terms are **fixed** for a given model. Only activations depend on batch size. So the maximum batch size is:

$$B_{\max} = \frac{M_{\text{GPU}} - 16\Psi}{L \cdot S \cdot H \cdot 34}$$

In practice, we also need ~10-15% overhead for PyTorch memory management, CUDA kernels, and fragmentation.

In [ ]:
#@title 🎧 Code Walkthrough: Calculator Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_calculator_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# The core memory calculator

def will_it_fit(
    num_params,
    num_layers,
    hidden_dim,
    num_heads,
    batch_size,
    seq_len,
    gpu_memory_gb,
    precision="mixed",
    optimizer="adam",
    activation_checkpointing=False,
    overhead_pct=0.12,
):
    """Comprehensive GPU memory calculator.

    Args:
        num_params: Total parameters (e.g., 7e9)
        num_layers: Number of transformer layers
        hidden_dim: Hidden dimension
        num_heads: Number of attention heads
        batch_size: Training batch size
        seq_len: Sequence length
        gpu_memory_gb: Available GPU memory in GB
        precision: 'fp32', 'fp16', or 'mixed'
        optimizer: 'adam', 'sgd', or 'adamw'
        activation_checkpointing: If True, assume sqrt(L) activation memory
        overhead_pct: PyTorch/CUDA overhead fraction (default 12%)

    Returns:
        Dictionary with detailed memory breakdown and fit verdict.
    """
    to_gb = lambda b: b / (1024**3)

    # --- Weights ---
    if precision == "fp32":
        weight_gb = to_gb(num_params * 4)
    else:
        weight_gb = to_gb(num_params * 2)

    # --- Gradients ---
    grad_gb = weight_gb  # same size as weights

    # --- Optimizer ---
    if optimizer in ["adam", "adamw"]:
        if precision == "mixed":
            opt_gb = to_gb(num_params * 12)  # master + m + v
        elif precision == "fp32":
            opt_gb = to_gb(num_params * 8)   # m + v (weights already fp32)
        else:
            opt_gb = to_gb(num_params * 8)   # m + v
    elif optimizer == "sgd":
        opt_gb = to_gb(num_params * 4)  # just momentum buffer
    else:
        opt_gb = to_gb(num_params * 8)  # default: assume 2 states

    # --- Activations ---
    bytes_per_activation = 34 + 5 * (num_heads * seq_len / hidden_dim)
    act_bytes = num_layers * batch_size * seq_len * hidden_dim * bytes_per_activation

    if activation_checkpointing:
        # Checkpointing saves activations only at sqrt(L) layers
        act_bytes = act_bytes * (num_layers ** 0.5) / num_layers

    act_gb = to_gb(act_bytes)

    # --- Totals ---
    model_gb = weight_gb + grad_gb + opt_gb  # fixed cost
    total_gb = model_gb + act_gb
    usable_gb = gpu_memory_gb * (1 - overhead_pct)
    overhead_gb = gpu_memory_gb * overhead_pct

    fits = total_gb <= usable_gb

    # --- Max batch size ---
    act_per_sample = num_layers * seq_len * hidden_dim * bytes_per_activation
    if activation_checkpointing:
        act_per_sample = act_per_sample * (num_layers ** 0.5) / num_layers
    remaining_bytes = (usable_gb - model_gb) * (1024**3)
    max_batch_size = max(0, int(remaining_bytes / act_per_sample))

    return {
        "weights_gb": weight_gb,
        "gradients_gb": grad_gb,
        "optimizer_gb": opt_gb,
        "activations_gb": act_gb,
        "model_fixed_gb": model_gb,
        "total_gb": total_gb,
        "overhead_gb": overhead_gb,
        "usable_gb": usable_gb,
        "gpu_memory_gb": gpu_memory_gb,
        "fits": fits,
        "max_batch_size": max_batch_size,
        "utilization_pct": (total_gb / usable_gb) * 100 if usable_gb > 0 else 999,
    }

print("will_it_fit() calculator ready!")

## 4. Let's Build It -- Component by Component

### The Pretty Printer

In [ ]:
#@title 🎧 Code Walkthrough: Printer Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_printer_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def print_report(result, model_name="Model", config_str=""):
    """Print a formatted memory report."""
    r = result

    verdict = "FITS" if r['fits'] else "DOES NOT FIT"
    verdict_emoji = "[OK]" if r['fits'] else "[X]"

    print(f"\n{'='*65}")
    print(f"  WILL IT FIT?  {model_name}")
    if config_str:
        print(f"  {config_str}")
    print(f"{'='*65}")
    print(f"")
    print(f"  {'Component':<30} {'Memory':>10} {'Share':>8}")
    print(f"  {'-'*48}")
    print(f"  {'Weights':<30} {r['weights_gb']:>8.2f} GB {r['weights_gb']/r['total_gb']*100:>6.1f}%")
    print(f"  {'Gradients':<30} {r['gradients_gb']:>8.2f} GB {r['gradients_gb']/r['total_gb']*100:>6.1f}%")
    print(f"  {'Optimizer States':<30} {r['optimizer_gb']:>8.2f} GB {r['optimizer_gb']/r['total_gb']*100:>6.1f}%")
    print(f"  {'Activations':<30} {r['activations_gb']:>8.2f} GB {r['activations_gb']/r['total_gb']*100:>6.1f}%")
    print(f"  {'-'*48}")
    print(f"  {'TOTAL REQUIRED':<30} {r['total_gb']:>8.2f} GB")
    print(f"  {'GPU Available':<30} {r['gpu_memory_gb']:>8.2f} GB")
    print(f"  {'Overhead (~12%)':<30} {r['overhead_gb']:>8.2f} GB")
    print(f"  {'Usable':<30} {r['usable_gb']:>8.2f} GB")
    print(f"")
    print(f"  Verdict: {verdict_emoji} {verdict}")
    print(f"  GPU utilization: {r['utilization_pct']:.1f}%")
    print(f"  Max batch size: {r['max_batch_size']}")
    print(f"{'='*65}")

    if not r['fits']:
        excess = r['total_gb'] - r['usable_gb']
        print(f"\n  Exceeds GPU by {excess:.1f} GB. Suggestions:")
        print(f"    1. Reduce batch size (current activations: {r['activations_gb']:.1f} GB)")
        if r['max_batch_size'] > 0:
            print(f"    2. Max batch size that fits: {r['max_batch_size']}")
        else:
            print(f"    2. Model params alone ({r['model_fixed_gb']:.1f} GB) exceed GPU!")
            print(f"       -> Need model parallelism or ZeRO")
        print(f"    3. Use activation checkpointing to reduce activation memory")
        print(f"    4. Use ZeRO to shard optimizer states across GPUs")

print("Report printer ready.")

In [ ]:
#@title 🎧 Code Walkthrough: Configs
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_configs.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Pre-defined model configurations

MODEL_CONFIGS = {
    "GPT-2 (124M)": {
        "num_params": 124e6, "num_layers": 12,
        "hidden_dim": 768, "num_heads": 12,
    },
    "GPT-2 XL (1.5B)": {
        "num_params": 1.5e9, "num_layers": 48,
        "hidden_dim": 1600, "num_heads": 25,
    },
    "LLaMA-7B": {
        "num_params": 7e9, "num_layers": 32,
        "hidden_dim": 4096, "num_heads": 32,
    },
    "LLaMA-13B": {
        "num_params": 13e9, "num_layers": 40,
        "hidden_dim": 5120, "num_heads": 40,
    },
    "LLaMA-70B": {
        "num_params": 70e9, "num_layers": 80,
        "hidden_dim": 8192, "num_heads": 64,
    },
    "Mixtral-8x7B (active)": {
        "num_params": 12.9e9, "num_layers": 32,
        "hidden_dim": 4096, "num_heads": 32,
    },
}

GPU_CONFIGS = {
    "T4 (16 GB)": 16,
    "RTX 3090 (24 GB)": 24,
    "RTX 4090 (24 GB)": 24,
    "A100 (40 GB)": 40,
    "A100 (80 GB)": 80,
    "H100 (80 GB)": 80,
}

print("Available model configs:", list(MODEL_CONFIGS.keys()))
print("Available GPU configs: ", list(GPU_CONFIGS.keys()))

In [ ]:
#@title 🎧 Code Walkthrough: Examples
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_examples.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Example 1: GPT-2 on a T4 (Colab free tier)

config = MODEL_CONFIGS["GPT-2 (124M)"]
result = will_it_fit(
    **config,
    batch_size=8,
    seq_len=1024,
    gpu_memory_gb=16,
    precision="mixed",
)

print_report(result, "GPT-2 (124M)", "B=8, S=1024, Mixed Precision, T4 16GB")

In [ ]:
# Example 2: LLaMA-7B on A100 80GB

config = MODEL_CONFIGS["LLaMA-7B"]
result = will_it_fit(
    **config,
    batch_size=4,
    seq_len=2048,
    gpu_memory_gb=80,
    precision="mixed",
)

print_report(result, "LLaMA-7B", "B=4, S=2048, Mixed Precision, A100 80GB")

In [ ]:
# Example 3: The OOM scenario from the article!
# LLaMA-7B, batch_size=4, on a 24GB GPU

config = MODEL_CONFIGS["LLaMA-7B"]
result = will_it_fit(
    **config,
    batch_size=4,
    seq_len=2048,
    gpu_memory_gb=24,
    precision="mixed",
)

print_report(result, "LLaMA-7B", "B=4, S=2048, Mixed Precision, RTX 3090 24GB")

print("\nThis is exactly the OOM scenario from the article!")
print("The model params alone (weights + grads + optimizer) exceed 24 GB.")

In [ ]:
#@title 🎧 Listen: Visual Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_visual_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### The Visual Report

In [ ]:
#@title 🎧 What to Look For: Visual Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_visual_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def visual_report(result, model_name="Model"):
    """Create a visual memory report."""
    r = result

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # --- Panel 1: Memory waterfall ---
    components = [
        ('Weights', r['weights_gb'], '#2196F3'),
        ('Gradients', r['gradients_gb'], '#4CAF50'),
        ('Optimizer', r['optimizer_gb'], '#FF5722'),
        ('Activations', r['activations_gb'], '#9C27B0'),
        ('Overhead', r['overhead_gb'], '#757575'),
    ]

    names = [c[0] for c in components]
    values = [c[1] for c in components]
    colors = [c[2] for c in components]
    cumulative = np.cumsum([0] + values[:-1])

    bars = axes[0].bar(names, values, bottom=cumulative, color=colors, width=0.6)

    # GPU limit line
    axes[0].axhline(y=r['gpu_memory_gb'], color='red', linestyle='--',
                     linewidth=2, label=f"GPU: {r['gpu_memory_gb']:.0f} GB")

    # Annotate each bar with size
    for bar, val, cum in zip(bars, values, cumulative):
        if val > 0.5:  # Only label if visible
            axes[0].text(bar.get_x() + bar.get_width()/2, cum + val/2,
                        f'{val:.1f} GB', ha='center', va='center',
                        fontsize=10, color='white', fontweight='bold')

    axes[0].set_ylabel('Memory (GB)', fontsize=12)
    axes[0].set_title(f'{model_name}: Memory Waterfall', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(axis='y', alpha=0.3)

    # --- Panel 2: Gauge-style utilization ---
    # Stacked horizontal bar
    used = r['total_gb']
    available = r['gpu_memory_gb']
    remaining = max(0, available - used)
    excess = max(0, used - available)

    if r['fits']:
        axes[1].barh(0, used, color='#4CAF50', height=0.5, label=f'Used: {used:.1f} GB')
        axes[1].barh(0, remaining, left=used, color='#E0E0E0', height=0.5, label=f'Free: {remaining:.1f} GB')
    else:
        axes[1].barh(0, available, color='#FF5722', height=0.5, label=f'GPU: {available:.0f} GB')
        axes[1].barh(0, excess, left=available, color='#B71C1C', height=0.5, label=f'Excess: {excess:.1f} GB')

    axes[1].set_xlim(0, max(used, available) * 1.15)
    axes[1].set_yticks([])
    axes[1].set_xlabel('Memory (GB)', fontsize=12)

    verdict = "FITS!" if r['fits'] else "DOES NOT FIT"
    color = '#4CAF50' if r['fits'] else '#FF5722'
    axes[1].set_title(f'Verdict: {verdict}', fontsize=16, fontweight='bold', color=color)
    axes[1].legend(fontsize=11, loc='center right')

    # Add max batch size annotation
    axes[1].text(0.5, -0.25, f'Max batch size: {r["max_batch_size"]}',
                transform=axes[1].transAxes, fontsize=14, ha='center',
                fontweight='bold')

    plt.tight_layout()
    plt.show()

# Show visual report for LLaMA-7B on A100
config = MODEL_CONFIGS["LLaMA-7B"]
result = will_it_fit(**config, batch_size=1, seq_len=2048, gpu_memory_gb=80, precision="mixed")
visual_report(result, "LLaMA-7B on A100 80GB (B=1)")

In [ ]:
#@title 🎧 Listen: Matrix Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_matrix_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### The Compatibility Matrix

Let us check which models fit on which GPUs.

In [ ]:
#@title 🎧 Code Walkthrough: Matrix Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_matrix_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Build a compatibility matrix: which models fit on which GPUs?

print("Compatibility Matrix: Model x GPU (mixed precision, B=1, S=2048)")
print("Max batch size shown. '-' means model params alone exceed GPU.")
print("=" * 90)

gpu_names = list(GPU_CONFIGS.keys())
header = f"{'Model':<25}" + "".join(f"{g:<12}" for g in gpu_names)
print(header)
print("-" * 90)

for model_name, config in MODEL_CONFIGS.items():
    row = f"{model_name:<25}"
    for gpu_name, gpu_mem in GPU_CONFIGS.items():
        result = will_it_fit(
            **config,
            batch_size=1,
            seq_len=2048,
            gpu_memory_gb=gpu_mem,
            precision="mixed",
        )
        if result['max_batch_size'] > 0:
            row += f"B<={result['max_batch_size']:<8}"
        else:
            row += f"{'  -':<12}"
    print(row)

print("\nNote: These are theoretical maximums. Actual batch sizes may be slightly lower")
print("due to memory fragmentation and PyTorch internals.")

In [ ]:
#@title 🎧 Before You Start: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Your Turn -- TODO Exercises

### TODO 1: Your GPU, Your Model

Use the calculator to check if you can train GPT-2 (124M) on your current Colab GPU. Find the maximum batch size with sequence length 512.

In [ ]:
#@title 🎧 Before You Start: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Check GPT-2 training feasibility on your current GPU

# Step 1: Detect your GPU memory (or set manually)
if torch.cuda.is_available():
    my_gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    my_gpu_name = torch.cuda.get_device_name(0)
else:
    my_gpu_gb = 16  # Assume T4
    my_gpu_name = "T4 (assumed)"

print(f"Your GPU: {my_gpu_name} ({my_gpu_gb:.1f} GB)")

# TODO: Call will_it_fit for GPT-2 with different batch sizes
# Find the maximum batch size that fits with seq_len=512

config = MODEL_CONFIGS["GPT-2 (124M)"]

# TODO: Try batch sizes 1, 2, 4, 8, 16, 32, 64 and find the largest that fits
# for bs in [1, 2, 4, 8, 16, 32, 64]:
#     result = will_it_fit(**config, batch_size=bs, seq_len=512,
#                          gpu_memory_gb=my_gpu_gb, precision="mixed")
#     fits = "FITS" if result['fits'] else "NO"
#     print(f"  B={bs:<4} -> {fits} (total: {result['total_gb']:.2f} GB)")

pass  # REPLACE WITH YOUR CODE

In [ ]:
#@title 🎧 Before You Start: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 2: Activation Checkpointing Impact

Activation checkpointing (gradient checkpointing) saves only $\sqrt{L}$ layers of activations instead of all $L$, recomputing the rest during the backward pass.

Use the `activation_checkpointing=True` parameter in `will_it_fit()` to see how much it helps for LLaMA-7B. How much larger can the batch size be?

In [ ]:
#@title 🎧 Before You Start: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Compare memory with and without activation checkpointing

config = MODEL_CONFIGS["LLaMA-7B"]

# TODO: Run will_it_fit with activation_checkpointing=False and =True
# for batch_size=1, seq_len=2048, on A100 80GB

# result_no_ckpt = will_it_fit(**config, batch_size=1, seq_len=2048,
#                               gpu_memory_gb=80, activation_checkpointing=False)
# result_ckpt = will_it_fit(**config, batch_size=1, seq_len=2048,
#                            gpu_memory_gb=80, activation_checkpointing=True)

# TODO: Print the comparison
# print(f"Without checkpointing: {result_no_ckpt['activations_gb']:.2f} GB activations, max B={result_no_ckpt['max_batch_size']}")
# print(f"With checkpointing:    {result_ckpt['activations_gb']:.2f} GB activations, max B={result_ckpt['max_batch_size']}")

pass  # REPLACE WITH YOUR CODE

In [ ]:
#@title 🎧 Before You Start: Todo3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_todo3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 3: Custom Model Configuration

Design a transformer model that fits on a T4 (16 GB) GPU for training with batch size 4 and sequence length 512. Find the maximum number of parameters you can use.

**Constraints:**
- Hidden dim must be divisible by num_heads
- Use mixed precision
- Must fit with B=4, S=512

In [ ]:
#@title 🎧 Before You Start: Todo3 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_todo3_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Design a model that fits on T4 with B=4, S=512

# Approach: Start with a small config and increase until it doesn't fit

# TODO: Try different configurations
# Suggestions to try:
# - hidden_dim: 512, 768, 1024, 1536, 2048
# - num_layers: 6, 12, 18, 24
# - num_heads: hidden_dim // 64 (common ratio)

# For each config, estimate num_params as:
# num_params ~ num_layers * (12 * hidden_dim^2 + 13 * hidden_dim) + vocab_size * hidden_dim
# (This is the standard transformer parameter count formula)

def estimate_params(num_layers, hidden_dim, vocab_size=32000):
    """Rough estimate of transformer parameter count."""
    # Per layer: 4 weight matrices of size [H, H] for attention + 2 FFN matrices
    per_layer = 12 * hidden_dim**2 + 13 * hidden_dim  # approx
    embeddings = vocab_size * hidden_dim
    return num_layers * per_layer + embeddings

# TODO: Search for the largest model that fits
# best_config = None
# for H in [512, 768, 1024, 1536, 2048]:
#     for L in [6, 12, 18, 24]:
#         A = H // 64
#         P = estimate_params(L, H)
#         result = will_it_fit(P, L, H, A, batch_size=4, seq_len=512,
#                              gpu_memory_gb=16, precision='mixed')
#         if result['fits']:
#             print(f"L={L}, H={H}, A={A}: {P/1e6:.0f}M params, {result['total_gb']:.2f} GB -> FITS")
#             best_config = (L, H, A, P)

pass  # REPLACE WITH YOUR CODE

In [ ]:
#@title 🎧 Transition: Heatmap Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_heatmap_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Putting It All Together

Let us create the ultimate visualization: a heatmap showing the maximum batch size for every model x GPU combination.

In [ ]:
#@title 🎧 What to Look For: Heatmap Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_heatmap_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Heatmap: max batch size for each model x GPU combination

model_names = list(MODEL_CONFIGS.keys())
gpu_names_short = [g.split('(')[0].strip() for g in GPU_CONFIGS.keys()]
gpu_mems = list(GPU_CONFIGS.values())

# Build the matrix
matrix = np.zeros((len(model_names), len(gpu_mems)))

for i, (model_name, config) in enumerate(MODEL_CONFIGS.items()):
    for j, gpu_mem in enumerate(gpu_mems):
        result = will_it_fit(
            **config,
            batch_size=1,
            seq_len=2048,
            gpu_memory_gb=gpu_mem,
            precision="mixed",
        )
        matrix[i, j] = result['max_batch_size']

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 7))

# Use log scale colors for better visibility
im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=max(matrix.flatten().max(), 1))

ax.set_xticks(range(len(gpu_names_short)))
ax.set_xticklabels(gpu_names_short, rotation=30, ha='right', fontsize=11)
ax.set_yticks(range(len(model_names)))
ax.set_yticklabels(model_names, fontsize=11)

# Add text annotations
for i in range(len(model_names)):
    for j in range(len(gpu_mems)):
        val = int(matrix[i, j])
        text = f"B={val}" if val > 0 else "X"
        color = 'black' if val > 5 else ('white' if val == 0 else 'black')
        ax.text(j, i, text, ha='center', va='center', fontsize=10,
                fontweight='bold', color=color)

ax.set_title('Max Batch Size: Model x GPU\n(Mixed Precision, S=2048)', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Max Batch Size', shrink=0.8)

plt.tight_layout()
plt.show()

print("Red/X = does not fit even with B=1 (need multi-GPU or ZeRO)")
print("Green = fits comfortably with larger batch sizes")

In [ ]:
#@title 🎧 Transition: Validation Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_validation_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Training and Results

Let us verify the calculator against actual GPU memory usage.

In [ ]:
#@title 🎧 Code Walkthrough: Validation Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_validation_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Validate the calculator against real GPU measurements

if torch.cuda.is_available():
    from torch.cuda.amp import autocast, GradScaler

    class TestTransformer(nn.Module):
        def __init__(self, vocab_size=10000, hidden_dim=512, num_layers=6, num_heads=8):
            super().__init__()
            self.embed = nn.Embedding(vocab_size, hidden_dim)
            self.layers = nn.ModuleList([
                nn.TransformerEncoderLayer(
                    d_model=hidden_dim, nhead=num_heads,
                    dim_feedforward=hidden_dim*4, batch_first=True
                )
                for _ in range(num_layers)
            ])
            self.head = nn.Linear(hidden_dim, vocab_size)

        def forward(self, x):
            x = self.embed(x)
            for layer in self.layers:
                x = layer(x)
            return self.head(x)

    # Test configurations
    test_configs = [
        (4, 128, "small batch"),
        (8, 128, "medium batch"),
        (16, 128, "large batch"),
        (4, 256, "long sequence"),
    ]

    hidden_dim = 512
    num_layers = 6
    num_heads = 8
    vocab_size = 10000

    print(f"Validation: TestTransformer (H={hidden_dim}, L={num_layers}, A={num_heads})")
    print(f"{'Config':<25} {'Predicted (GB)':<18} {'Measured (GB)':<18} {'Error'}")
    print("-" * 75)

    for bs, sl, desc in test_configs:
        # Prediction
        num_params = sum(p.numel() for p in TestTransformer(vocab_size, hidden_dim, num_layers, num_heads).parameters())
        pred = will_it_fit(
            num_params, num_layers, hidden_dim, num_heads,
            bs, sl, gpu_memory_gb=80, precision="mixed"
        )

        # Actual measurement
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        model = TestTransformer(vocab_size, hidden_dim, num_layers, num_heads).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=1e-4)
        scaler = GradScaler()

        x = torch.randint(0, vocab_size, (bs, sl)).to(device)
        t = torch.randint(0, vocab_size, (bs, sl)).to(device)

        opt.zero_grad()
        with autocast(dtype=torch.float16):
            out = model(x)
            loss = nn.functional.cross_entropy(out.view(-1, vocab_size), t.view(-1))
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        measured_gb = torch.cuda.max_memory_allocated() / 1e9

        del model, opt, scaler, x, t
        torch.cuda.empty_cache()

        error_pct = abs(pred['total_gb'] - measured_gb) / measured_gb * 100
        print(f"B={bs}, S={sl} ({desc}){'':<8} {pred['total_gb']:<18.3f} {measured_gb:<18.3f} {error_pct:.1f}%")

    print("\nNote: Some error is expected due to PyTorch overhead, temporary buffers, etc.")
else:
    print("GPU required for validation. The theoretical calculator is still useful without a GPU.")

In [ ]:
#@title 🎧 Listen: Summary Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_summary_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Final Output

In [ ]:
#@title 🎧 Code Walkthrough: Summary Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_summary_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("=" * 65)
print("SUMMARY: Will It Fit? GPU Memory Calculator")
print("=" * 65)
print()
print("The Complete Formula (mixed precision + Adam):")
print("  M_total = 16P + L * B * S * H * 34 bytes")
print("          = (fixed model cost) + (variable activation cost)")
print()
print("Maximum Batch Size:")
print("  B_max = (GPU_mem * 0.88 - 16P) / (L * S * H * 34)")
print()
print("Quick Reference Card:")
print("-" * 65)
print(f"  {'Model':<20} {'Fixed Cost':<15} {'Act/sample':<15} {'Min GPU'}")
print("-" * 65)
for name, config in MODEL_CONFIGS.items():
    r = will_it_fit(**config, batch_size=1, seq_len=2048, gpu_memory_gb=1000, precision='mixed')
    fixed = r['model_fixed_gb']
    act_1 = r['activations_gb']
    min_gpu = fixed + act_1 + fixed * 0.12
    print(f"  {name:<20} {fixed:>8.1f} GB    {act_1:>8.1f} GB    {min_gpu:>6.0f} GB")
print()
print("When It Does Not Fit:")
print("  1. Reduce batch size (cheapest fix)")
print("  2. Activation checkpointing (trades compute for memory)")
print("  3. ZeRO-1/2/3 (shard optimizer/gradients/weights across GPUs)")
print("  4. Tensor parallelism (split model across GPUs)")
print("  5. Pipeline parallelism (split layers across GPUs)")

In [ ]:
#@title 🎧 Narration: Reflection
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_25_reflection.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Reflection and Next Steps

**Questions to think about:**
1. Why does the calculator include a 12% overhead? What real-world factors contribute to this?
2. For inference (not training), which components are NOT needed? How much memory does inference require?
3. How would the memory budget change if you used AdaFactor (which uses less state than Adam) instead of Adam?

**What comes next:**
Now that you understand exactly where GPU memory goes, the next pods in this course will teach you how to **reduce** each component:
- **Pod 3 (Recomputation + Accumulation + DP):** Reduce activations with gradient checkpointing, simulate large batches with gradient accumulation, and distribute training with data parallelism
- **Pod 4 (Tensor Parallelism):** Split model weights across GPUs
- **Pod 5 (ZeRO):** Shard optimizer states, gradients, and weights

Every technique you will study in this course targets one or more of the four memory components you now understand.